In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [8]:
BASE_DIR = Path().resolve().parent
DATA_DIR = BASE_DIR / "data" / "processed"

df_input = pd.read_csv(DATA_DIR / "input_metrics_processed.csv")
df_orders = pd.read_csv(DATA_DIR / "orders_metrics_processed.csv")

df_input.head()

,COUNTRY,CITY,ZONE,ZONE_TYPE,ZONE_PRIORITIZATION,METRIC,NW_AGO,MVALUE
0,EC,Quito,San Martin de Porras,Non Wealthy,Not Prioritized,Retail SST > SS CVR,8,0.602339
1,BR,Bauru,FS Parque Jardim Europa/Vila Riachuelo - BAU,Non Wealthy,Not Prioritized,Restaurants SST > SS CVR,8,0.857544
2,MX,Mexicali,MXL_Universidad,Wealthy,Prioritized,Retail SST > SS CVR,8,0.900823
3,CL,Santiago De Chile,Colina,Non Wealthy,Not Prioritized,Retail SST > SS CVR,8,0.798946
4,MX,Guadalajara,Valle Real,Wealthy,High Priority,Gross Profit UE,8,3.914477


In [9]:
df_input.columns = [c.upper() for c in df_input.columns]
df_orders.columns = [c.upper() for c in df_orders.columns]

df_input = df_input.fillna(0)
df_orders = df_orders.fillna(0)

“Top 5 zones with highest % Lead Penetration this week”

In [10]:
latest_week = df_input["NW_AGO"].min()

result_filtering = (
    df_input[
        (df_input["METRIC"] == "Lead Penetration") &
        (df_input["NW_AGO"] == latest_week)
    ]
    .sort_values("MVALUE", ascending=False)
    .head(5)
)

result_filtering

,COUNTRY,CITY,ZONE,ZONE_TYPE,ZONE_PRIORITIZATION,METRIC,NW_AGO,MVALUE
111614,EC,Quito,Sur de Quito,Non Wealthy,High Priority,Lead Penetration,0,393.9
112140,EC,Quito,Valle de los Chillos,Non Wealthy,High Priority,Lead Penetration,0,158.6
103981,EC,Quito,Antiguo Aeropuerto,Non Wealthy,Prioritized,Lead Penetration,0,141.3
111892,EC,Quito,Vicentina- Floresta,Non Wealthy,Prioritized,Lead Penetration,0,87.8
112228,EC,Guayaquil,Via la Costa Oeste,Wealthy,Prioritized,Lead Penetration,0,57.2


“Compare Perfect Orders between Wealthy and Non-Wealthy zones in Mexico”

In [13]:
df_mexico = df_input[df_input["COUNTRY"] == "MX"]

result_comparison = (
    df_mexico[df_mexico["METRIC"] == "Perfect Orders"]
    .groupby("ZONE_TYPE")["MVALUE"]
    .mean()
    .reset_index()
)

result_comparison

,ZONE_TYPE,MVALUE
0,Non Wealthy,0.843464
1,Wealthy,0.893146


Evolution of Gross Profit UE in Chapinero (last 8 weeks)

In [17]:
result_trend = (
    df_input[
        (df_input["ZONE"] == "Chapinero") &
        (df_input["METRIC"] == "Gross Profit UE")
    ]
    .sort_values("NW_AGO")
    .tail(8)
)

result_trend

,COUNTRY,CITY,ZONE,ZONE_TYPE,ZONE_PRIORITIZATION,METRIC,NW_AGO,MVALUE
50147,CO,Bogota,Chapinero,Wealthy,Prioritized,Gross Profit UE,5,3.192055
38522,CO,Bogota,Chapinero,Wealthy,Prioritized,Gross Profit UE,5,3.192055
37574,CO,Bogota,Chapinero,Wealthy,Prioritized,Gross Profit UE,6,3.168068
25949,CO,Bogota,Chapinero,Wealthy,Prioritized,Gross Profit UE,6,3.168068
25001,CO,Bogota,Chapinero,Wealthy,Prioritized,Gross Profit UE,7,3.063769
13376,CO,Bogota,Chapinero,Wealthy,Prioritized,Gross Profit UE,7,3.063769
803,CO,Bogota,Chapinero,Wealthy,Prioritized,Gross Profit UE,8,2.904238
12428,CO,Bogota,Chapinero,Wealthy,Prioritized,Gross Profit UE,8,2.904238


Average Lead Penetration by country

In [18]:
result_aggregation = (
    df_input[df_input["METRIC"] == "Lead Penetration"]
    .groupby("COUNTRY")["MVALUE"]
    .mean()
    .reset_index()
    .sort_values("MVALUE", ascending=False)
)

result_aggregation

,COUNTRY,MVALUE
5,EC,13.947092
7,PE,0.868386
3,CO,0.526125
2,CL,0.265480
0,AR,0.252086
6,MX,0.226119
1,BR,0.186694
8,UY,0.111674
4,CR,0.052992


Zones with high Lead Penetration but low Perfect Orders”

In [20]:
latest_week = df_input["NW_AGO"].min()

pivot = (
    df_input[df_input["NW_AGO"] == latest_week]
    .pivot_table(
        index="ZONE",
        columns="METRIC",
        values="MVALUE"
    )
    .fillna(0)
)

result_multi = pivot[
    (pivot["Lead Penetration"] > pivot["Lead Penetration"].quantile(0.75)) &
    (pivot["Perfect Orders"] < pivot["Perfect Orders"].quantile(0.25))
]

result_multi

METRIC,% PRO Users Who Breakeven,% Restaurants Sessions With Optimal Assortment,Gross Profit UE,Lead Penetration,MLTV Top Verticals Adoption,Non-Pro PTC > OP,Perfect Orders,Pro Adoption (Last Week Status),Restaurants Markdowns / GMV,Restaurants SS > ATC CVR,Restaurants SST > SS CVR,Retail SST > SS CVR,Turbo Adoption
ZONE,,,,,,,,,,,,,
Campo Grande Expansión,0.000000,0.000000,-8.224153,0.310644,0.004348,0.423929,0.802107,0.111111,0.024377,0.248471,0.572980,0.671712,0.0
Catamarca,0.018182,0.000000,-1.314050,0.405601,0.098621,0.634610,0.705946,0.196262,0.167431,0.304089,0.846219,0.857205,0.0
Chaullabamba,0.000000,0.000000,0.000000,13.600000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
Cidade Satélite,0.000000,0.000000,-5.776146,2.000000,0.000000,0.075000,0.000000,0.285714,0.000000,0.061538,0.800000,0.900000,0.0
Curauma,0.236842,0.000000,0.187638,0.317590,0.040984,0.669537,0.817669,0.361345,0.177416,0.500078,0.869911,0.780913,0.0
FS Petrópolis/Centro/Jardim Blumenau BNU,0.000000,0.000000,-0.616014,0.390476,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.600000,0.0
Floresta,0.125000,0.009996,-0.484898,34.600000,0.086723,0.745772,0.792756,0.250000,0.232207,0.349291,0.819101,0.728101,0.0
Jardim Ouro Verde,0.000000,0.000000,4.253436,4.900000,0.000000,0.516190,0.000000,0.428571,0.000000,0.620357,0.875000,0.853333,0.0
Leonor,0.000000,0.000000,-2.008844,2.300000,0.000000,0.313333,0.542411,0.500000,0.000000,0.410714,0.856623,0.840000,0.0


Zones with highest growth in orders (last 5 weeks)

In [23]:
df_orders_sorted = df_orders.sort_values("NW_AGO")

growth_results = []

for city, group in df_orders_sorted.groupby("CITY"):
    group = group.sort_values("NW_AGO").tail(5)

    if len(group) >= 2:
        growth = (
            group["ORDERS"].iloc[-1] - group["ORDERS"].iloc[0]
        ) / (group["ORDERS"].iloc[0] + 1e-6)

        growth_results.append((city, growth))

result_inference = (
    pd.DataFrame(growth_results, columns=["CITY", "GROWTH"])
    .sort_values("GROWTH", ascending=False)
    .head(10)
)

result_inference

,CITY,GROWTH
277,Santos,844000000.0
51,Cartago,814000000.0
11,Aracaju,122000000.0
247,Rio Verde,5000000.0
295,Taubate,3000000.0
146,Lagos De Moreno,3000000.0
26,Bauru,3000000.0
101,Foz Do Iguaçu,3000000.0
126,Indaiatuba,3000000.0
177,Mogi Das Cruzes,2000000.0
